# Qwen2.5-3B-Instruct + QLoRA — customer support agent

Target: Kaggle T4 (16 GB) or Colab T4. Runtime → GPU.

Before running: this notebook is the *only* supported training path. The local repo machine has no GPU.
After training, record the produced `run_manifest.json` in `EXPERIMENT_LOG.md`.

In [ ]:
import os, sys, subprocess
REPO = '/kaggle/working/support-llm-3b'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/Sahilgumber7/customer-support-engine.git', REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, 'src')
sys.path.insert(0, '.')
!pip install -q -r training/requirements-train.txt
!nvidia-smi

In [ ]:
# Regenerate the SFT data if the JSONL files are not in the repo (they are gitignored).
from pathlib import Path
if not Path('data/sft/train.jsonl').exists():
    subprocess.run([sys.executable, '-m', 'data_gen.generate', '--variants', '4'], check=True)
print(sum(1 for _ in open('data/sft/train.jsonl')), 'train rows')
print(sum(1 for _ in open('data/sft/dev.jsonl')), 'dev rows')

In [ ]:
# 20-step smoke test: catches config/shape errors before spending GPU hours.
!python training/train_qlora.py --max-steps 20 --out /kaggle/working/smoke

In [ ]:
# Full run: 2 epochs, ~1.5-3 h on a T4.
!python training/train_qlora.py --epochs 2 --out /kaggle/working/qwen3b-support-lora

In [ ]:
# Merge + quantise to Q4_K_M for CPU serving.
!python training/merge_and_export.py --adapter /kaggle/working/qwen3b-support-lora --out /kaggle/working/merged

## After the run

1. Download `qwen3b-support-lora/run_manifest.json` and `merged/qwen2.5-3b-support-q4_k_m.gguf`.
2. Append the real numbers (loss, wall time, GPU, peak VRAM, hashes) to `EXPERIMENT_LOG.md` as `EXP-001`.
3. Put the GGUF at `models/qwen2.5-3b-support-q4_k_m.gguf` locally.
4. Run `python -m eval.harness --backend llamacpp` and compare against the baseline JSON.

Do not edit `EVALUATION_RESULTS/` by hand.